In [1]:
Drug1_name = 'Vemurafenib'
Drug2_name = 'MITHRAMYCIN'
cell_line_name = 'T47D'

In [2]:
import pandas as pd 

import pandas as pd 

drug_f_dir = '../../data/DrugComb/Process/cid_features_Roberta.pkl'
kegg_dir = "../../data/DrugComb/Process/DepMap_kegg.csv"
gene_dir = "../../data/DrugComb/Process/MOCO_feature_Depmap_ProteinCodingGene.pkl"
protein_dir = "../../data/DrugComb/Process/TCPA_CCLE_RPPA500.tsv"
meta_dir = "../../data/DrugComb/Process/Metabolomics_subsetted.csv"
model_dir = "../../data/DrugComb/Process/Model.csv"
geneEffect_dir = "../../data/DrugComb/Process/MOCO_feature_Depmap_geneEffect.pkl"
ssGSEA_dir = "../../data/DrugComb/Process/MOCO_feature_Depmap_ssGSEA.pkl"
geneDependency_dir = "../../data/DrugComb/Process/MOCO_feature_Depmap_geneDependency.pkl"
methylation_dir = "../../data/DrugComb/Process/MOCO_feature_Depmap_methylation.pkl"
CNV_dir = "../../data/DrugComb/Process/MOCO_feature_Depmap_CNV.pkl"
mutation_dir = "../../data/DrugComb/Process/Depmap/Depmap_mutation_top_var_256.pkl"

cids_dir = "../../data/DrugComb/Process/DrugComb_all_witch_CID_data_name2_CID_dict.pkl"



kegg = pd.read_csv(kegg_dir,index_col=0,header=0)
protein = pd.read_table(protein_dir,sep='\t',index_col=0,header=0)
gene = pd.read_pickle(gene_dir)
pc_ls = [i.split("_")[0] for i in protein.index.to_list()]
protein.index = pc_ls

meta = pd.read_csv(meta_dir,index_col=0)
cell_model = pd.read_csv(model_dir,index_col=0)
ssGSEA = pd.read_pickle(ssGSEA_dir)
geneEffect = pd.read_pickle(geneEffect_dir)
geneDependency = pd.read_pickle(geneDependency_dir)
methylation = pd.read_pickle(methylation_dir)
CNV = pd.read_pickle(CNV_dir)
mutation = pd.read_pickle(mutation_dir)

# 定义替换规则
index_replacement_rules = {
    'COLO699': 'CHL1DM',
    'D341MED': 'D341Med'
}

protein = protein.rename(index=index_replacement_rules)







In [3]:
drug_cids = pd.read_pickle(cids_dir)
drug_f = pd.read_pickle(drug_f_dir)
geneT = gene.T
metaT = meta.T
proteinT = protein.T
keggT = kegg.T
geneEffectT = geneEffect.T
ssGSEAT = ssGSEA.T
geneDependencyT = geneDependency.T
methylationT = methylation.T
CNVT = CNV.T
mutationT = mutation.T

model_cell_name_ls = cell_model["StrippedCellLineName"].unique().tolist()
import torch


no_in_gene = []
no_in_meta = []
no_in_kegg = []
no_in_protein = []
no_in_geneEffect = []
no_in_ssGSEA = []
no_in_geneDependency = []
no_in_methylation = []
no_in_CNV = []
no_in_mutation = []
def pro_data(drug_row,drug_col,cell_line_name):
    data = {}
    data["drug_row"] = drug_row
    data["drug_col"] = drug_col
    data["cell_line_name"] = cell_line_name
    data["drug_f1"] = drug_f[drug_cids[drug_row]]
    data["drug_f2"] = drug_f[drug_cids[drug_col]]
    cell_name = cell_line_name
    if cell_name in model_cell_name_ls:
        cl_id = cell_model[cell_model["StrippedCellLineName"]==cell_name].index.tolist()[0]
    else:
        cl_id = None
    if cl_id in metaT.columns.tolist():
        data["meta_f"] = metaT[cl_id]
    else:
        no_in_meta.append(cell_name)
        data["meta_f"] = metaT.mean(axis=1)
    if cl_id in kegg.columns.tolist():
        data["kegg_f"] = kegg[cl_id]
    else:
        no_in_kegg.append(cell_name)
        data["kegg_f"] = kegg.mean(axis=1)

    if cl_id in geneT.columns.tolist():
        data["gene_f"] = geneT[cl_id]
    else:
        no_in_gene.append(cell_name)
        data["gene_f"] = geneT.mean(axis=1)
    
    if cell_name in proteinT.columns.tolist():
        data["protein_f"] = proteinT[cell_name]
    else:
        no_in_protein.append(cell_name)
        data["protein_f"] = proteinT.mean(axis=1)
    if cl_id in geneEffectT.columns.tolist():
        data["geneEffect_f"] = geneEffectT[cl_id]
    else:
        no_in_geneEffect.append(cell_name)
        data["geneEffect_f"] = geneEffectT.mean(axis=1)
    if cl_id in ssGSEAT.columns.tolist():
        data["ssGSEA_f"] = ssGSEAT[cl_id]
    else:
        no_in_ssGSEA.append(cell_name)
        data["ssGSEA_f"] = ssGSEAT.mean(axis=1)
    if cl_id in geneDependencyT.columns.tolist():
        data["geneDependency_f"] = geneDependencyT[cl_id]
    else:
        no_in_geneDependency.append(cell_name)
        data["geneDependency_f"] = geneDependencyT.mean(axis=1)
    if cl_id in methylationT.columns.tolist():
        data["methylation_f"] = methylationT[cl_id]
    else:
        no_in_methylation.append(cell_name)
        data["methylation_f"] = methylationT.mean(axis=1)
    if cl_id in CNVT.columns.tolist():
        data["CNV_f"] = CNVT[cl_id]
    else:
        no_in_CNV.append(cell_name)
        data["CNV_f"] = CNVT.mean(axis=1)
    if cl_id in mutationT.columns.tolist():
        data["mutation_f"] = mutationT[cl_id]
    else:
        no_in_mutation.append(cell_name)
        data["mutation_f"] = mutationT.mean(axis=1)
    return data

In [4]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader





# 2. 优化 Dataset 类
class mydata1(Dataset):
    def __init__(self, data):
        self.data = data
        # 预处理数据
        self._preprocess_data()
        
    def _preprocess_data(self):
        # 提前将数据转换为张量并移到CPU内存
        self.processed_data = []
        for item in self.data:
            processed_item = {
                "drug_f1": torch.as_tensor(item['drug_f1'], dtype=torch.float32),
                "drug_f2": torch.as_tensor(item['drug_f2'], dtype=torch.float32),
                "molt5_f1": torch.as_tensor(item['molt5_f1'], dtype=torch.float32),
                "molt5_f2": torch.as_tensor(item['molt5_f2'], dtype=torch.float32),
                "gene_f": torch.as_tensor(item['gene_f'], dtype=torch.float32),
                "protein_f": torch.as_tensor(item['protein_f'], dtype=torch.float32),
                "kegg_f": torch.as_tensor(item['kegg_f'], dtype=torch.float32),
                "meta_f": torch.as_tensor(item['meta_f'], dtype=torch.float32),
                "zip": torch.as_tensor([item['zip']], dtype=torch.float32),
                "loewe": torch.as_tensor([item['loewe']], dtype=torch.float32),
                "hsa": torch.as_tensor([item['hsa']], dtype=torch.float32),
                "bliss": torch.as_tensor([item['bliss']], dtype=torch.float32),
                "geneEffect_f": torch.as_tensor(item['geneEffect_f'], dtype=torch.float32),
                "ssGSEA_f": torch.as_tensor(item['ssGSEA_f'], dtype=torch.float32),
                "geneDependency_f": torch.as_tensor(item['geneDependency_f'], dtype=torch.float32),
                "methylation_f": torch.as_tensor(item['methylation_f'], dtype=torch.float32),
                "CNV_f": torch.as_tensor(item['CNV_f'], dtype=torch.float32),
                "mutation_f": torch.as_tensor(item['mutation_f'], dtype=torch.float32),
                #"HSA": torch.as_tensor([item['synergy'][0]], dtype=torch.float32)
            }
            self.processed_data.append(processed_item)

    def __len__(self):
        return len(self.processed_data)
    
    def __getitem__(self, index):
        return self.processed_data[index]

In [5]:
from torch.utils.data import ConcatDataset

test_fold = [0]
data_fold_ls = [0,1,2,3,4]
tran_fold = [i for i in data_fold_ls if i not in test_fold]

combined_train = torch.load('../../data/SHAP/drugcomb_backgroud_data_shap_kmeans_K-5120_sbatch.pt')
train_dataset = torch.tensor(combined_train.data,dtype=torch.float32)

In [9]:
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=len(train_dataset),
    shuffle=True)


In [10]:
for backgroud_data in train_dataloader:
    break

'\n\nfor batch in train_dataloader:\n    drug_f1 = batch["drug_f1"][:,0:768]\n    drug_f2 = batch["drug_f2"][:,0:768]\n    synergy_zip = batch["zip"][:,0]\n    gene_f = batch["gene_f"][:,0:256]\n    kegg_f = batch["kegg_f"][:,0:178]\n    protein_f = batch["protein_f"][:,0:447]\n    meta_f = batch["meta_f"][:,0:225]\n    geneEffect_f = batch["geneEffect_f"]\n    ssGSEA_f = batch["ssGSEA_f"]\n    geneDependency_f = batch["geneDependency_f"]\n    methylation_f = batch["methylation_f"]\n    CNV_f = batch["CNV_f"]\n    mutation_f = batch["mutation_f"]\n    break\n\nbackgroud_data = torch.cat([drug_f1,drug_f2,gene_f,kegg_f,protein_f,meta_f,geneEffect_f,ssGSEA_f,geneDependency_f,methylation_f,CNV_f,mutation_f],\n                            dim=1)\n'

In [13]:
import shap
import pandas as pd

import torch
from torch import nn
import torch.nn.functional as F

class EmbeddingLayer(nn.Module):
    def __init__(self, feature_length,hid_dim):
        super().__init__()
        self.embedding = nn.Linear(feature_length, hid_dim)
        self.head_dim = hid_dim

    def forward(self, x):
        #print(x.shape[0])
        return self.embedding(x).view(x.shape[0],1,self.head_dim)


class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout):
        super().__init__()
        assert hid_dim % n_heads == 0
        self.hid_dim = hid_dim
        self.n_heads = n_heads
        self.head_dim = hid_dim // n_heads
        self.fc_q = nn.Linear(hid_dim, hid_dim)
        self.fc_k = nn.Linear(hid_dim, hid_dim)
        self.fc_v = nn.Linear(hid_dim, hid_dim)
        self.fc_o = nn.Linear(hid_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim]))

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        Q = self.fc_q(query)
        K = self.fc_k(key)
        V = self.fc_v(value)
        Q = Q.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale.to(Q.device)
        # energy = [batch size, n heads, query len, key len]
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        self.attention = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(self.attention), V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(batch_size, -1, self.hid_dim)
        x = self.fc_o(x)
        return x


#@save
class AddNorm(nn.Module):
    """残差连接后进行层规范化"""
    def __init__(self, normalized_shape, dropout, **kwargs):
        super(AddNorm, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(normalized_shape)

    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)

#@save
class EncoderBlock(nn.Module):
    """Transformer编码器块"""
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 norm_shape, num_heads,
                 dropout, use_bias=False, **kwargs):
        super(EncoderBlock, self).__init__(**kwargs)
        self.attention = MultiHeadAttentionLayer(
            num_hiddens, num_heads, dropout)
        self.addnorm1 = AddNorm(norm_shape, dropout)

    def forward(self, Q,K,V):
        return self.addnorm1(Q, self.attention(Q, K, V))

#@save
class TransformerEncoder(nn.Module):
    """Transformer编码器"""
    def __init__(self, feature_size, key_size, query_size, value_size,
                 num_hiddens, norm_shape,num_heads, num_layers, dropout, use_bias=False, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.embedding = EmbeddingLayer(feature_size, num_hiddens)
        self.blks = nn.Sequential()
        for i in range(num_layers):
            self.blks.add_module("block"+str(i),
                EncoderBlock(key_size, query_size, value_size, num_hiddens,
                             norm_shape,num_heads, dropout, use_bias))

    def forward(self, Q,K,V,*args):
        # 因为位置编码值在-1和1之间，
        # 因此嵌入值乘以嵌入维度的平方根进行缩放，
        # 然后再与位置编码相加。
        Q = self.embedding(Q)
        K = self.embedding(K)
        V = self.embedding(V)
        self.attention_weights = [None] * len(self.blks)
        for i, blk in enumerate(self.blks):
            X = blk(Q,K,V)
            self.attention_weights[
                i] = blk.attention
        return X
    
# 示例网络架构优化
class ImprovedSelfAttentionNetwork(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        # 增加网络深度和复杂性
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads,batch_first=True,dropout=0.3) 
            for _ in range(8)  # 增加注意力层数
        ])
        
         #添加残差连接和层归一化
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(8)
        ])
        
        # 增加非线性变换
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )

        self.mlps = nn.ModuleList([
           self.mlp for _ in range(8)
        ])
        
    def forward(self, Q,K,V):
        # 实现多层注意力和残差连接
        for attention, norm, mlp in zip(self.attention_layers, self.layer_norms, self.mlps):
        #for attention in self.attention_layers:
            residual = Q
            Q, _ = attention(Q, K, V)
            Q = norm(Q + residual)
            Q = mlp(Q)
        return Q


#--------------------------------------------------------------------------------
class Predictor(nn.Module):
    def __init__(self, ) -> None:
        super().__init__()
        #feature_size = input_size
        # Drug - cell -Drug cancentration
        feature_size = 768
        Drop_rate = 0.3
        self.D12C_encoder = ImprovedSelfAttentionNetwork(hidden_dim=768, num_heads=8)
        
        self.CD12_encoder = ImprovedSelfAttentionNetwork(hidden_dim=768, num_heads=8)
        
        self.D122_encoder = ImprovedSelfAttentionNetwork(hidden_dim=768, num_heads=8)

        self.CD_encoder = ImprovedSelfAttentionNetwork(hidden_dim=768, num_heads=8)

        self.DC_encoder = ImprovedSelfAttentionNetwork(hidden_dim=768, num_heads=8)

        self.DCD_encoder = ImprovedSelfAttentionNetwork(hidden_dim=768, num_heads=8)

        self.gene_p = nn.Sequential(
            nn.Linear(256, 256)
        )

        self.kegg_p = nn.Sequential(
            nn.Linear(178, 256)
        )

        self.protein_p = nn.Sequential(
            nn.Linear(447, 256)
        )

        self.meta_p = nn.Sequential(
            nn.Linear(225, 256)
        )

        self.geneEffect_p = nn.Sequential(
            nn.Linear(256, 256)
        )

        self.ssGSEA_p = nn.Sequential(
            nn.Linear(256, 256)
        )

        self.geneDependency_p = nn.Sequential(
            nn.Linear(256, 256)
        )

        self.methylation_p = nn.Sequential(
            nn.Linear(256, 256)
        )

        self.CNV_p = nn.Sequential(
            nn.Linear(256, 256)
        )

        self.mutation_p = nn.Sequential(
            nn.Linear(256, 256)
        )

        self.drug_p = nn.Sequential(
            nn.Linear(768, 768)
        )

        self.gene_f = nn.Sequential(
            nn.Linear(2048+256*2, 1024),
            nn.Linear(1024, 768)
        )
        
        
        self.bliss0 = nn.Sequential(
            nn.Linear(768*12, 2048),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(1024, 256),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

        self.bliss1 = nn.Sequential(
            nn.Linear(768*12, 4068),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(4068, 2048),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(2048, 768),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(768, 256),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(256, 1),
        )

        self.bliss2 = nn.Sequential(
            nn.Linear(768*12, 2048),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(2048, 768),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(768, 128),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(128, 1),
        )

        self.var = nn.Sequential(
            nn.Linear(768*12, 2048),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(1024, 256),
            nn.Dropout(Drop_rate),
            nn.ReLU(),
            nn.Linear(256, 1),
        )


    def forward(self,input_data,step = None):
        drug_f1 = input_data[:, :768]
        drug_f2 = input_data[:, 768:(768*2)]
        gene_f = input_data[:, (768*2):(768*2+256)]
        kegg_f = input_data[:, (768*2+256):(768*2+256+178)]
        protein_f = input_data[:, (768*2+256+178):(768*2+256+178+447)]
        meta_f = input_data[:, (768*2+256+178+447):(768*2+256+178+447+225)]
        geneEffect_f = input_data[:, (768*2+256+178+447+225):(768*2+256+178+447+225+256)]
        ssGSEA_f = input_data[:, (768*2+256+178+447+225+256):(768*2+256+178+447+225+256*2)]
        geneDependency_f = input_data[:, (768*2+256+178+447+225+256*2):(768*2+256+178+447+225+256*3)]
        methylation_f = input_data[:, (768*2+256+178+447+225+256*3):(768*2+256+178+447+225+256*4)]
        CNV_f = input_data[:, (768*2+256+178+447+225+256*4):(768*2+256+178+447+225+256*5)]
        mutation_f = input_data[:, (768*2+256+178+447+225+256*5):(768*2+256+178+447+225+256*6)]


        drug_f1 = self.drug_p(drug_f1).unsqueeze(1)
        drug_f2 = self.drug_p(drug_f2).unsqueeze(1)
        gene_f = self.gene_p(gene_f)
        kegg_f = self.kegg_p(kegg_f)
        protein_f = self.protein_p(protein_f)
        meta_f = self.meta_p(meta_f)
        geneEffect_f = self.geneEffect_p(geneEffect_f)
        ssGSEA_f = self.ssGSEA_p(ssGSEA_f)
        geneDependency_f = self.geneDependency_p(geneDependency_f)
        methylation_f = self.methylation_p(methylation_f)
        CNV_f = self.CNV_p(CNV_f)
        mutation_f = self.mutation_p(mutation_f)
        batch_size = drug_f1.size(0)
        

        gene_f1 = self.gene_f(torch.cat((gene_f,kegg_f,protein_f,meta_f,geneEffect_f,ssGSEA_f,geneDependency_f,methylation_f,CNV_f,mutation_f),dim=1)).unsqueeze(1)
        D12C_f_1 = self.D12C_encoder(drug_f1,drug_f2,gene_f1).reshape(batch_size,-1)
        D12C_f_2 = self.D12C_encoder(drug_f2,drug_f1,gene_f1).reshape(batch_size,-1)
        CD12_f_1 = self.CD12_encoder(gene_f1,drug_f1,drug_f2).reshape(batch_size,-1)
        CD12_f_2 = self.CD12_encoder(gene_f1,drug_f2,drug_f1).reshape(batch_size,-1)
        D122_f_1 = self.D122_encoder(drug_f1,drug_f2,drug_f2).reshape(batch_size,-1)
        D122_f_2 = self.D122_encoder(drug_f2,drug_f1,drug_f1).reshape(batch_size,-1)
        CD_f_1 = self.CD_encoder(gene_f1,drug_f1,drug_f1).reshape(batch_size,-1)
        CD_f_2 = self.CD_encoder(gene_f1,drug_f2,drug_f2).reshape(batch_size,-1)
        DC_f_1 = self.DC_encoder(drug_f1,gene_f1,gene_f1).reshape(batch_size,-1)
        DC_f_2 = self.DC_encoder(drug_f2,gene_f1,gene_f1).reshape(batch_size,-1)
        DCD_f_1 = self.DCD_encoder(drug_f1,gene_f1,drug_f2).reshape(batch_size,-1)
        DCD_f_2 = self.DCD_encoder(drug_f2,gene_f1,drug_f1).reshape(batch_size,-1)

        in1 = torch.cat((D12C_f_1,D12C_f_2,CD12_f_1,CD12_f_2,D122_f_1,D122_f_2,CD_f_1,CD_f_2,DC_f_1,DC_f_2,DCD_f_1,DCD_f_2),dim=1)
        bliss1 = (self.bliss0(in1) + self.bliss1(in1) + self.bliss2(in1))/3
        var = nn.functional.softplus(self.var(in1))
        return bliss1

In [14]:
#import synergy_model

zip_model = Predictor()

zip_pth_dir = "../../models/AXIS.pth"
zip_model.load_state_dict(torch.load(zip_pth_dir,map_location="cpu"))



<All keys matched successfully>

## 进行Shap分析

In [15]:
device2 = 'cpu'
zip_model.to(device2)
zip_model.eval()


Predictor(
  (D12C_encoder): ImprovedSelfAttentionNetwork(
    (attention_layers): ModuleList(
      (0-7): 8 x MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
      )
    )
    (layer_norms): ModuleList(
      (0-7): 8 x LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
    (mlp): Sequential(
      (0): Linear(in_features=768, out_features=1536, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=1536, out_features=768, bias=True)
    )
    (mlps): ModuleList(
      (0-7): 8 x Sequential(
        (0): Linear(in_features=768, out_features=1536, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.3, inplace=False)
        (3): Linear(in_features=1536, out_features=768, bias=True)
      )
    )
  )
  (CD12_encoder): ImprovedSelfAttentionNetwork(
    (attention_layers): ModuleList(
      (0-7): 8 x MultiheadAttention(
        (out_proj): NonDynamicallyQuantiz

In [16]:
import shap
import torch
import numpy as np

# 假设你的模型已加载
# zip_model.load_state_dict(torch.load(f"{output_dir}/result/final/zip_model_{step}.pth"))
zip_model.eval()

backgroud_data.requires_grad_()
explainer = shap.DeepExplainer(zip_model, backgroud_data.to(device2))



In [18]:
feature_name_ls = []

for i in range(768):
    feature_name_ls.append('drug_f1_' + str(i))
for i in range(768):
    feature_name_ls.append('drug_f2_' + str(i))
for i in range(256):
    feature_name_ls.append('gene_f_' + str(i))
for i in range(178):
    feature_name_ls.append('kegg_f_' + str(i))
for i in range(447):
    feature_name_ls.append('protein_f_' + str(i))
for i in range(225):
    feature_name_ls.append('meta_f_' + str(i))
for i in range(256):
    feature_name_ls.append('geneEffect_f_' + str(i))
for i in range(256):
    feature_name_ls.append('ssGSEA_f_' + str(i))
for i in range(256):
    feature_name_ls.append('geneDependency_f_' + str(i))
for i in range(256):
    feature_name_ls.append('methylation_f_' + str(i))
for i in range(256):
    feature_name_ls.append('CNV_f_' + str(i))
for i in range(256):
    feature_name_ls.append('mutation_f_' + str(i))



In [19]:

def get_explain_samp(drug_1,drug_2,cell_name,device2):
    explain_samp =  pro_data(drug_1,drug_2,cell_name)
    drug_f1 =  torch.as_tensor(explain_samp["drug_f1"], dtype=torch.float32).to(device2)
    drug_f2 =  torch.as_tensor(explain_samp["drug_f2"], dtype=torch.float32).to(device2)
    #print(drug_f2)
    gene_f =  torch.as_tensor(explain_samp["gene_f"], dtype=torch.float32).to(device2)
    kegg_f =  torch.as_tensor(explain_samp["kegg_f"], dtype=torch.float32).to(device2)
    protein_f =  torch.as_tensor(explain_samp["protein_f"], dtype=torch.float32).to(device2)
    meta_f =  torch.as_tensor(explain_samp["meta_f"], dtype=torch.float32).to(device2)
    geneEffect_f =  torch.as_tensor(explain_samp["geneEffect_f"], dtype=torch.float32).to(device2)
    ssGSEA_f =  torch.as_tensor(explain_samp["ssGSEA_f"], dtype=torch.float32).to(device2)
    geneDependency_f =  torch.as_tensor(explain_samp["geneDependency_f"], dtype=torch.float32).to(device2)
    methylation_f =  torch.as_tensor(explain_samp["methylation_f"], dtype=torch.float32).to(device2)
    CNV_f =  torch.as_tensor(explain_samp["CNV_f"], dtype=torch.float32).to(device2)
    mutation_f =  torch.as_tensor(explain_samp["mutation_f"], dtype=torch.float32).to(device2)

    explain_data = torch.cat([drug_f1,drug_f2,gene_f,kegg_f,protein_f,meta_f,geneEffect_f,ssGSEA_f,geneDependency_f,methylation_f,CNV_f,mutation_f],
                                dim=0)
    return explain_data

In [ ]:
import matplotlib.pyplot as plt

# 选择一个样本进行解释

sample_data = get_explain_samp(Drug1_name,Drug2_name,cell_line_name,device2)  # 取第一个样本，形状为 (1, 特征总数)
sample_data.requires_grad_()
# 计算 SHAP 值
shap_values = explainer.shap_values(sample_data.reshape(1, -1), check_additivity=False) 
idx = 0
fdf = pd.DataFrame({"shap": shap_values[idx].reshape(-1), "feature": sample_data.cpu().detach().numpy()})
fdf['feature_name'] = feature_name_ls



/tmp/ipykernel_84445/170025273.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  kegg_f =  torch.as_tensor(explain_samp["kegg_f"], dtype=torch.float32).to(device2)
/tmp/ipykernel_84445/170025273.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  protein_f =  torch.as_tensor(explain_samp["protein_f"], dtype=torch.float32).to(device2)
/tmp/ipykernel_84445/170025273.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  meta_f =  torch

In [22]:
fdf.to_csv(f'../../data/SHAP/Mid/mid_shap_test_{Drug1_name}_{Drug2_name}_{cell_line_name}.csv',index=False)